In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.gold
COMMENT 'Capa Gold procesadoss'


In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_ubication_proyecto")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_ubication (
  ubicacion_id INT,
  departamento_descripcion STRING,
  municipio_descripcion STRING
)
""")

In [0]:
import pandas as pd

# 1. Leer desde Silver
df = spark.table("workspace.silver.predios_registro").toPandas()

# 2. Seleccionar columnas necesarias
ubicacion = (
    df.reindex(
        columns=[
            "departamento_descripcion",
            "municipio_descripcion"
        ]
    )

)

# 3. Limpieza
dim_ubication = (
    ubicacion
    .dropna(subset=["departamento_descripcion", "municipio_descripcion"])
    .drop_duplicates()
    .sort_values(["departamento_descripcion", "municipio_descripcion"], ignore_index=True)
)

# 4. Crear ID (clave sustituta)
dim_ubication["ubicacion_id"] = dim_ubication.index + 1

# 5. Convertir a Spark
df_spark = spark.createDataFrame(dim_ubication)